# Benchmark Quantum Adder

**Quantum arithmetic** is a central ingredient in many quantum algorithms. Quantum circuits that manipulate encoded integers or numerical variables coherently rely on reversible arithmetic primitives such as addition, comparison, and multiplication. These building blocks appear prominently in Shor’s algorithm, where modular arithmetic is essential, and also in oracle-based settings related to Grover-style search, where arithmetic can be part of the problem encoding. In practice, arithmetic subroutines often contribute significantly to circuit depth and overall resource cost.

Among these primitives, the **quantum adder** is one of the simplest and most canonical examples. It implements the reversible transformation of one register by adding the value of another, making it a natural starting point for studying quantum arithmetic. It is also a basic component in more advanced arithmetic circuits, including multipliers, comparators, accumulators, and modular routines.

We benchmark an out-of-place addition by a constant:
$$
|x\rangle_n|0\rangle_n \rightarrow |x\rangle_n|x+c\, (\bmod 2^n) \rangle_n,
$$
where $c$ is a given integer.

***

### The model
We start with a uniform superposition for $|x\rangle$, and add the constant $c=2^{\lfloor n/2 \rfloor}-1$:
$$
\frac{1}{\sqrt{2^n}}\sum_{x=0}^{2^n-1}|x\rangle_n|0\rangle_n \rightarrow \frac{1}{\sqrt{2^n}}\sum_{x=0}^{2^n-1}|x\rangle_n|x+c \, (\bmod 2^n)\rangle_n.
$$

### The Scoring function
We measure the probabilty of getting a correct result:
$$
{\rm Score = }\sum_{y,x=0}^{2^n-1} P\left(y= x+c \, (\bmod 2^n)\right),
$$
where $y$ is the measured value of the second variable.

***
***

## Defining a `BenchmarkExample` for an Adder

In [1]:
import asyncio
import datetime
import sys
from pathlib import Path

sys.path.insert(0, "..")

from benchmark import BenchmarkExample
from collector import ResultCollector
from hardware import HardwareRunner
from hardwares_preferences import HARDWARES

from classiq import *

In [2]:
ADDER_DESCRIPTION = Path("../descriptions/adder.tex").read_text(encoding="utf-8")

In [3]:
class AdderExample(BenchmarkExample):
    def __init__(self, num_qubits: int):
        super().__init__(
            name="adder",
            num_qubits=num_qubits,
            report_family_title="Adder",
            report_family_description=ADDER_DESCRIPTION,
            constraints=Constraints(optimization_parameter="width"),
        )

    def create_main(self) -> callable:
        @qfunc
        def main(y: Output[QNum[self.num_qubits]], x: Output[QNum[self.num_qubits]]):
            allocate(y)
            allocate(x)
            hadamard_transform(x)
            y ^= x + 2 ** (y.size // 2) - 1

        return main

    async def submit(self, qprog: QuantumProgram) -> str:
        with ExecutionSession(qprog) as es:
            job = es.submit_sample()
            return job.id

    async def score(self, job_id):
        job = ExecutionJob.from_id(job_id)
        result = await job.result_async()
        df = result[0].value.dataframe

        mask = (
            (df["x"] + 2 ** (self.num_qubits // 2) - 1 - df["y"]) % 2**self.num_qubits
        ) == 0

        exec_minutes = (job.end_time - job.start_time).total_seconds() / 60.0

        return {
            "score": df.loc[mask, "probability"].sum(),
            "execution_time": exec_minutes,
        }

***
***
## Benchmarking a 4-qubits adder

Define a specific example on 4 qubits (note that the models are optimized for width):

In [4]:
example = AdderExample(num_qubits=4)
example.show()

Quantum program link: https://platform.classiq.io/circuit/3BP0Wsn4g1HuAyPV8BTanBpNsE2


Define the number of shots and other hyperparameters:

In [5]:
common_config = {
    "max_timeout": 5e5,  # value is in seconds. Equals a bit more than 5 days."
    "num_shots": 1000,
}

### Choosing on which backend to run 

Define `HardwareRunner` for each backend. Here we choose one machine for IBM, one for IonQ, as well as Classiq simulators for reference.

*(The list of runners can be edited after defining the `ResultCollector`.)*

In [6]:
classiq_runners = [
    HardwareRunner("Classiq", backend_name, **common_config)
    for backend_name in HARDWARES["Classiq"][0:2]
]

ionq_runners = [
    HardwareRunner("IonQ", backend_name, **common_config)
    for backend_name in HARDWARES["IonQ"][0:1]
]


ibm_runners = [
    HardwareRunner(
        "IBM Quantum",
        backend_name,
        **common_config,
        backend_kwargs={
            "access_token": "PUT YOUR TOKEN HERE",
            "channel": "PUT YOUR CHANNEL HERE",
            "instance_crn": "PUT YOUR INSTANCE_CRN HERE",
        },
    )
    for backend_name in HARDWARES["IBM Quantum"][0:1]
]

runners = [
    *classiq_runners,
    # *ionq_runners,
    # *ibm_runners,
]

In [7]:
print("Running for Backends:")
print(
    *[(runner.backend_service_provider, runner.backend_name) for runner in runners],
    sep="\n"
)

Running for Backends:
('Classiq', 'simulator')
('Classiq', 'simulator_statevector')


### Set a `ResultCollector` with a file name fixed for the current example

In [8]:
FILENAME = example.default_results_filename

In [9]:
collector = ResultCollector(FILENAME, build_each_time=True)

In [10]:
print(
    "=" * 10
    + f"  {example.name}-{example.num_qubits} ({datetime.datetime.now()})   "
    + "=" * 10
)
await asyncio.gather(*[collector.run(runner, example) for runner in runners]);

==========  adder-4 (2026-03-24 20:03:54.351737)   ==========
2026-03-24 20:03:58.279474: Submit adder-4 for Classiq - simulator
2026-03-24 20:03:59.712889: Completed adder-4 for Classiq - simulator --> Score {'score': 1.0, 'execution_time': 0.013318799999999999}
** Report updated: adder-4 for Classiq - simulator


In [11]:
await collector.print_status()

========== (2026-03-24 20:04:00.827904)   ==========
adder-4 | IonQ - qpu.forte-1 | COMPLETED | score=0.8770 | time=15.14 min
adder-4 | IBM Quantum - ibm_kingston | COMPLETED | score=0.2040 | time=395.17 min
adder-4 | Classiq - simulator_statevector | COMPLETED | score=1.0000 | time=0.01 min
adder-4 | Classiq - simulator | COMPLETED | score=1.0000 | time=0.01 min
